In [4]:
import os, sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from nbafuns import *
from nba_api.stats.endpoints import leaguedashplayerptshot, playerdashptshots, leaguedashteamptshot

data_DIR = "../data/shots/"
box_DIR = "../data/box/"

In [5]:
season = 2024
stats = leaguegamelog.LeagueGameLog(
    player_or_team_abbreviation="T",
    season=season,
    season_type_all_star="Playoffs",
)
df = stats.get_data_frames()[0]
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], format="%Y-%m-%d")
df.to_parquet(box_DIR + "NBA_Box_T_" + "Base" + "_" + str(season) + "_PS.parquet")

In [ ]:
season = 2024
df0 = get_box(PT="T",measure="Base",cum=False,seasons=[season],playoffs=True)
datesl = df0["game_date"].unique()
dfr = pd.read_parquet(data_DIR + f"NBA_Shots_Def_Dist_{season}_PS.parquet")
datesr = dfr["game_date"].unique()
dates = list(set(datesl)-set(datesr))
# dates = datesl

In [18]:
closest_def = ['0-2 Feet - Very Tight', '2-4 Feet - Tight', '4-6 Feet - Open', '6+ Feet - Wide Open']

In [19]:
dfa = [dfr]
for date in tqdm(dates):
    d = date.strftime("%Y-%m-%d")
    for b in closest_def:
        stats = leaguedashteamptshot.LeagueDashTeamPtShot(
            league_id="00",
            season=get_ss(season),
            season_type_all_star="Playoffs",
            close_def_dist_range_nullable=b,
            date_from_nullable=date,
            date_to_nullable=date,

        ).get_data_frames()
        df1 = stats[0]
        df1["closest_def"] = b
        df1["game_date"] = date
        df1["season"] = season + 1
        dfa.append(df1)
        time.sleep(0.6)
with pd.option_context("future.no_silent_downcasting", True):                    
    dfa1 = [df2.fillna(0).infer_objects(copy=False) for df2 in dfa if not df2.empty]
df2 = pd.concat(dfa1)
df2.columns = map(str.lower,df2.columns)
df2 = df2.reset_index(drop=True)
df2.to_parquet(data_DIR + f"NBA_Shots_Def_Dist_{season}_PS.parquet")

100%|██████████| 22/22 [01:44<00:00,  4.76s/it]


ValueError: Cannot create parquet dataset with duplicate column names (Index(['team_id', 'team_name', 'team_abbreviation', 'gp', 'g', 'fga_frequency',
       'fgm', 'fga', 'fg_pct', 'efg_pct', 'fg2a_frequency', 'fg2m', 'fg2a',
       'fg2_pct', 'fg3a_frequency', 'fg3m', 'fg3a', 'fg3_pct', 'closest_def',
       'game_date', 'team_id', 'team_name', 'team_abbreviation', 'gp', 'g',
       'fga_frequency', 'fgm', 'fga', 'fg_pct', 'efg_pct', 'fg2a_frequency',
       'fg2m', 'fg2a', 'fg2_pct', 'fg3a_frequency', 'fg3m', 'fg3a', 'fg3_pct',
       'season'],
      dtype='object'))

In [ ]:
df01 = df0[['team_id','game_date','game_id','matchup','wl','season']]

In [ ]:
df3 = pd.merge(df01,df2,on=['team_id','game_date'])

In [ ]:
dist = '6+ Feet - Wide Open'
df4 = df3.query(f"closest_def == '{dist}'")
df4["home"]=~df4["matchup"].str.contains("@")
df4 = df4.sort_values(by=["game_date","game_id"])


In [ ]:
df5 = df4[['game_date', 'game_id','season','team_id','team_name','wl','fg3m','fg3a','fg3_pct','home']]

In [ ]:
df5g= df5.groupby("game_id")
df6_1 = df5g.nth(0)
df6_2 = df5g.nth(1)
cols1 = ['game_date', 'game_id', 'season']
cols2 = ['team_id','team_name','wl','fg3m', 'fg3a', 'fg3_pct','home']
df6_1.columns = cols1 + [s + "1" for s in cols2]
df6_2.columns = cols1 + [s + "2" for s in cols2]
df6 = pd.merge(df6_1, df6_2, on=cols1)
df7_1 = df5g.nth(1)
df7_2 = df5g.nth(0)
df7_1.columns = cols1 + [s + "1" for s in cols2]
df7_2.columns = cols1 + [s + "2" for s in cols2]
df7 = pd.merge(df7_1, df7_2, on=cols1)
df8 = pd.concat([df6,df7])
df8 = df8.query("home1 == True").reset_index(drop=True)


In [ ]:
df9 = df8.drop(columns = ['game_date', 'season', 'team_id1', 'home1', 'team_id2',  'home2','wl2'])
# df9 = df9[['game_id', 'team_name1','team_name2', 'wl1', 'fg3m1', 'fg3a1', 'fg3_pct1','fg3m2', 'fg3a2', 'fg3_pct2']]
df9 = df9[['game_id', 'team_name1','team_name2', 'wl1', 'fg3m1','fg3m2',]]
df9 = df9.rename(columns={"wl1":"wl"})

In [ ]:
df9["diff"] = df9['fg3m1'] - df9['fg3m2']
df9["w"] = np.where(df9["wl"] == "W",1,0)

In [ ]:
p = (
    ggplot(df9)
    + aes(x="diff",y="w")
    + geom_point()
    + theme_idv
)
p